<a href="https://colab.research.google.com/github/marikima/Levitador-ia-esp32/blob/main/Modelo_Llama3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Con este código se crea la base de datos para Llama 3.2 con 500 ejemplos.

import json
import random

#datos extraídos del informe
posiciones_conocidas = [14.0, 18.0, 20.0, 22.0, 24.0, 26.0, 30.0, 33.0]
pwms_conocidos = [1023, 1017, 1015, 1012, 1010, 1008, 999, 993]

#archivo que vamos a crear
nombre_archivo = "dataset_levitador.jsonl"

print("Creando base de datos para Llama 3.2...")

with open(nombre_archivo, "w", encoding="utf-8") as archivo:
    #acá se generan 500 ejemplos variados para que la IA aprenda
    for i in range(500):
        #se elige un objetivo (setpoint) aleatorio de la lista
        indice_objetivo = random.randint(0, len(posiciones_conocidas)-1)
        setpoint = posiciones_conocidas[indice_objetivo]
        pwm_ideal = pwms_conocidos[indice_objetivo]

        #acá simulamos dónde está la bola actualmente (un poco arriba o un poco abajo)
        posicion_actual = round(setpoint + random.uniform(-5.0, 5.0), 1)

        #acá se determina qué debe decir la IA
        if posicion_actual < setpoint:
            explicacion = f"La bola está en {posicion_actual} cm, por debajo del objetivo de {setpoint} cm. Necesitamos subirla aumentando la potencia."
        elif posicion_actual > setpoint:
            explicacion = f"La bola está en {posicion_actual} cm, por encima del objetivo de {setpoint} cm. Necesitamos bajarla reduciendo la potencia."
        else:
            explicacion = f"La bola está exactamente en el objetivo de {setpoint} cm. Mantenemos la potencia estable."

        #acá cree la estructura que Llama 3.2 entiende
        ejemplo = {
            "instruction": "Eres el controlador de IA de un levitador neumático. Tu objetivo es indicar el valor PWM correcto para llevar la bola al setpoint deseado.",
            "input": f"Setpoint: {setpoint} cm. Posición actual del sensor: {posicion_actual} cm.",
            "output": f"{explicacion} El PWM adecuado según la curva de calibración es {pwm_ideal}."
        }

        #se guarda en el archivo
        archivo.write(json.dumps(ejemplo, ensure_ascii=False) + "\n")

print(f"¡Listo! Se ha creado el archivo {nombre_archivo} con 500 ejemplos.")

Creando base de datos para Llama 3.2...
¡Listo! Se ha creado el archivo dataset_levitador.jsonl con 500 ejemplos.


In [ ]:
#Acá instaleré Unsloth (Máquina virtual) y librerías
%%capture
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade transformers tokenizers trl peft accelerate bitsandbytes xformers

In [ ]:
#En esta celda ejecuto un código para instalar el Llama 3.2
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True #se comprime el modelo

print("Descargando Llama 3.2... (Esto puede tomar un par de minutos)")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct", #aquí especifico el modelo
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("¡Modelo descargado y listo!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Descargando Llama 3.2... (Esto puede tomar un par de minutos)
==((====))==  Unsloth 2026.5.6: Fast Llama patching. Transformers: 5.9.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


¡Modelo descargado y listo!


In [ ]:
#En esta celda preparé el modelo para aprender (usé LoRA)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Modelo configurado con LoRA, exitosamente!!.")

[transformers] Unsloth 2026.5.6 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Modelo configurado con LoRA, exitosamente!!.


In [ ]:
#Acá cargo el archivo con los 500 ejemplos que se hizo al principio
from datasets import load_dataset

# Cargamos el archivo que ya tienes en Colab
dataset = load_dataset("json", data_files="dataset_levitador.jsonl", split="train")

# Le damos el formato que la IA necesita para leer
alpaca_prompt = """A continuación hay una instrucción que describe una tarea, junto con una entrada que proporciona más contexto. Escribe una respuesta que complete adecuadamente la solicitud.

### Instrucción:
{}

### Entrada:
{}

### Respuesta:
{}"""

EOS_TOKEN = tokenizer.eos_token # Le indica a la IA dónde termina la respuesta

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("¡Base de datos cargada y formateada!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

¡Base de datos cargada y formateada!


In [ ]:
#Acá hago el entrenamiento
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Definimos la configuración directamente como lo exige la versión más nueva
configuracion = SFTConfig(
    packing = True,              #Se empaquetan los textos
    max_seq_length = 2048,       #Longitud máxima permitida
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,              #Pasos de entrenamiento (prueba rápida)
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",          #Evita errores de servidores externos
    push_to_hub = False,         #Evita que intente subirlo a internet
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    dataset_num_proc = 2,
    args = configuracion,
)

print("Iniciando entrenamiento... Se verá una tabla con números bajando (Loss)")
trainer_stats = trainer.train()
print("¡ENTRENAMIENTO TERMINADO!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Iniciando entrenamiento... Verás una tabla con números bajando (Loss)


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 42 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.911999
2,2.910571
3,2.859583
4,2.652215
5,2.381148
6,2.054866
7,1.683930
8,1.366915
9,1.071060
10,0.825528


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


¡ENTRENAMIENTO TERMINADO!


In [ ]:
#En esta celda se puede probar al nuevo modelo

FastLanguageModel.for_inference(model) #Acá le decimos a la IA que ya no va a estudiar, sino a responder

#Hacemos la pregunta
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Eres el controlador de IA de un levitador neumático. Tu objetivo es indicar el valor PWM correcto para llevar la bola al setpoint deseado.", # instruction
        "Setpoint: 26.0 cm. Posición actual del sensor: 18.0 cm.", # input
        "", # output (Lo dejamos en blanco porque esto lo tiene que responder la IA)
    )
], return_tensors = "pt").to("cuda")

print("Pensando respuesta...\n")
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)

# Imprimimos lo que generó
texto_completo = tokenizer.batch_decode(outputs)[0]
# Filtramos solo la respuesta final
respuesta = texto_completo.split("### Respuesta:\n")[1].replace("<|eot_id|>", "")

print("--- RESPUESTA DE LA IA ---")
print(respuesta)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pensando respuesta...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces

--- RESPUESTA DE LA IA ---
La bola está en 18.0 cm, por debajo del objetivo de 26.0 cm. Necesitamos subirla aumentando la potencia. El PWM adecuado según la curva de calibración es 1008.
